In [55]:
import pandas as pd

df = pd.read_parquet("/home/postyr/Assessing-AI-Scoring-Accuracy-for-De-Novo-Proteins/Analysis/Data/native_metrics_with_af3_plddt.parquet")

df_csv = pd.read_csv("/home/postyr/Assessing-AI-Scoring-Accuracy-for-De-Novo-Proteins/results/AdaptyvBio/contest_2_round_2_metrics_summary.csv")

chosen_seq = "SRLEEIVRETLRENGFSEEEIEHITWHFLLLWGSLERTYPHWSEEERVRRAAEYTQMMHPEITPEMVEKLVSRV"
chosen_method = "SeedFold"   # e.g. AF3, Boltz-2, SeedFold, Protenix

native = df[
    (df["binder_sequence"] == chosen_seq) &
    (df["method"] == chosen_method) &
    (df["rank"] == 1)
]


In [44]:
merged = df.merge(df_csv[["sequence", "design_class", "binding"]], how="left", left_on="binder_sequence", right_on="sequence")
merged = merged.drop(columns=["sequence"])

In [45]:
plot = merged[merged["design_class"].isin(["Miniprotein", "Peptide"])].copy()
final = plot[
    (plot["rank"] == 1) &
    (plot["binding"] == False) 
].copy()

final.columns.to_list()

['path',
 'file_type',
 'method',
 'complex_name',
 'prediction_id',
 'rank',
 'target_chain_id',
 'binder_chain_id',
 'target_length',
 'binder_length',
 'binder_sequence',
 'target_sequence',
 'score',
 'score_name',
 'json_path',
 'pkl_path',
 'npz_path',
 'parse_success',
 'error_message',
 'af2_plddt_mean',
 'af2_plddt_median',
 'af2_plddt_min',
 'af2_plddt_max',
 'af2_ptm',
 'af2_iptm',
 'af3_ranking_score',
 'af3_ptm',
 'af3_iptm',
 'af3_fraction_disordered',
 'af3_has_clash',
 'boltz2_confidence_score',
 'boltz2_ptm',
 'boltz2_iptm',
 'boltz2_ligand_iptm',
 'boltz2_protein_iptm',
 'boltz2_complex_plddt',
 'boltz2_complex_iplddt',
 'boltz2_complex_pde',
 'boltz2_complex_ipde',
 'boltz2_source_json',
 'chai1_aggregate_score',
 'chai1_ptm',
 'chai1_iptm',
 'chai1_has_inter_chain_clashes',
 'helixfold3_ptm',
 'helixfold3_iptm',
 'helixfold3_has_clash',
 'helixfold3_mean_plddt',
 'of3_avg_plddt',
 'of3_gpde',
 'of3_iptm',
 'of3_ptm',
 'of3_disorder',
 'of3_has_clash',
 'of3_sample_r

In [25]:
chai_ex = final[
    (final["method"] == "Chai-1") &
    (final["binder_sequence"] == chosen_seq)
]

chai_ex[["binder_sequence", "complex_name", "rank", "prediction_id"]]

,binder_sequence,complex_name,rank,prediction_id
6071,SRLEEIVRETLRENGFSEEEIEHITWHFLLLWGSLERTYPHWSEEE...,steady-ram-lotus,1,pred.model_idx_4


In [4]:
plddt_col = "af3_plddt_mean"

af3_nonbinders_sorted = final.sort_values(
    by=plddt_col,
    ascending=False
)

top_example = af3_nonbinders_sorted.iloc[0]
top_example[["binder_sequence", "af3_plddt_mean", "rank", "complex_name"]]

binder_sequence    SRLEEIVRETLRENGFSEEEIEHITWHFLLLWGSLERTYPHWSEEE...
af3_plddt_mean                                             90.350405
rank                                                               1
complex_name                            reed.harrison.model_56_119_0
Name: 3617, dtype: object

In [5]:
af3_nonbinders_sorted[
    ["binder_sequence", plddt_col]
].head(10)

,binder_sequence,af3_plddt_mean
3617,SRLEEIVRETLRENGFSEEEIEHITWHFLLLWGSLERTYPHWSEEE...,90.350405
2595,SAADAAAAAAYAEYAAETTEHAKKALEAYEKGDLGTMVENALLAET...,90.328801
3030,SEVEGIRKSAEQLSKQPDAKTQLENLLQALKDLGAPEEAIKVAQEA...,90.318670
2261,SSFSSYCLHGTPFYESSLNKWSCRCDKGYYGPRCEYKDLL,90.159116
2285,YCLHGTPVYISSLNKWSCVCDKGWYGERCEFRDL,89.964360
2600,DAAARHAHRQRVLAESAFDIGRAIHLGLGPDKEEESEPFYAAVAAY...,89.950878
2270,SSFLTYCLHGTPKYESSLNKWSCVCDPGWYGERCEFKDLL,89.941996
3492,SDIEDVAKLYLGTIERLLERRGLTEKLPEVRAKIEELVEKGDLDGL...,89.880909
2543,MTPEELFEEFVETYFSIQDKLEKTLGPDNPKAKKVYEEMMKIWMRF...,89.802051
2222,SELEEIVRETLRENGFSDEEIEHITWHFLLLWGSLERTYPHWSREE...,89.766041


In [6]:
chosen_seq = top_example["binder_sequence"]
print(chosen_seq)

SRLEEIVRETLRENGFSEEEIEHITWHFLLLWGSLERTYPHWSEEERVRRAAEYTQMMHPEITPEMVEKLVSRV


In [42]:
[col for col in native.columns if "iptm" in col.lower()]

['af2_iptm',
 'af3_iptm',
 'boltz2_iptm',
 'boltz2_ligand_iptm',
 'boltz2_protein_iptm',
 'chai1_iptm',
 'helixfold3_iptm',
 'of3_iptm',
 'of3_chain_pair_iptm__(A, B)',
 'of3_bespoke_iptm__(A, B)',
 'protenix_iptm',
 'seedfold_iptm',
 'seedfold_ligand_iptm',
 'seedfold_protein_iptm',
 'seedfold_pair_chains_iptm__0__0',
 'seedfold_pair_chains_iptm__0__1',
 'seedfold_pair_chains_iptm__1__0',
 'seedfold_pair_chains_iptm__1__1']

In [59]:
iptm_value = native["seedfold_iptm"].iloc[0]
print(iptm_value)

0.41700461506843567


In [61]:
plddt_value = native["seedfold_complex_plddt"].iloc[0]
print(plddt_value)

0.8863463997840881
